## Setup

In [1]:
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2
from torchvision import datasets

In [2]:
from pathlib import Path

root_dir = Path('D:/ICIFAR10')
classes = {"airplane":0, "automobile":1, "bird":2, "cat":3, "deer":4, "dog":5, "frog":6, "horse":7, "ship":8, "truck":9}

## Google Colab

In [ ]:
!unzip train.zip -d train
!unzip valid.zip -d valid
!unzip cifar-10-python.zip -d cifar-10-python

In [ ]:
root_dir = Path('/content')

## Folder Structure

In [3]:
train_dir = root_dir / 'train'
validation_dir = root_dir / 'valid'

In [4]:
for dir in root_dir.iterdir():
    print(dir)

D:\ICIFAR10\cifar-10-python
D:\ICIFAR10\icifar10.pth
D:\ICIFAR10\icifar10_sampler.pth
D:\ICIFAR10\sample_data
D:\ICIFAR10\test
D:\ICIFAR10\train
D:\ICIFAR10\train_labels.csv
D:\ICIFAR10\valid
D:\ICIFAR10\validation_labels.csv


In [5]:
for dir in train_dir.iterdir():
    print(dir)

D:\ICIFAR10\train\airplane
D:\ICIFAR10\train\automobile
D:\ICIFAR10\train\bird
D:\ICIFAR10\train\cat
D:\ICIFAR10\train\deer
D:\ICIFAR10\train\dog
D:\ICIFAR10\train\frog
D:\ICIFAR10\train\horse
D:\ICIFAR10\train\ship
D:\ICIFAR10\train\truck


## Class distribution on training

In [6]:
train_sizes = []
for cls in classes:
    dir = train_dir / cls
    tot = 0
    for arquivo in dir.iterdir():
        tot += 1
    train_sizes.append(tot)

train_size = sum(train_sizes)
print(f'Total: {train_size}')
for i, cls in enumerate(classes):
    print(f'[{cls}] {train_sizes[i]} ({(100 * train_sizes[i] / train_size):.2f}%)')

Total: 29009
[airplane] 4500 (15.51%)
[automobile] 3080 (10.62%)
[bird] 2385 (8.22%)
[cat] 3497 (12.05%)
[deer] 2718 (9.37%)
[dog] 205 (0.71%)
[frog] 2273 (7.84%)
[horse] 3877 (13.36%)
[ship] 2033 (7.01%)
[truck] 4441 (15.31%)


## Class distribution on validation

In [7]:
validation_sizes = []
for cls in classes:
    dir = validation_dir / cls
    tot = 0
    for arquivo in dir.iterdir():
        tot += 1
    validation_sizes.append(tot)

validation_size = sum(validation_sizes)
print(f'Total: {validation_size}')
for i, cls in enumerate(classes):
    print(f'[{cls}] {validation_sizes[i]} ({(100 * validation_sizes[i] / validation_size):.2f}%)')

Total: 3222
[airplane] 500 (15.52%)
[automobile] 342 (10.61%)
[bird] 265 (8.22%)
[cat] 388 (12.04%)
[deer] 302 (9.37%)
[dog] 23 (0.71%)
[frog] 252 (7.82%)
[horse] 431 (13.38%)
[ship] 226 (7.01%)
[truck] 493 (15.30%)


## Dataset

In [8]:
import pandas as pd

labels_train = root_dir / 'train_labels.csv'
labels_validation = root_dir / 'validation_labels.csv'

In [9]:
from torchvision.io import decode_image

class ICIFAR10(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        row = self.img_labels.iloc[idx]

        img_path = self.img_dir / str(row['label'])
        img_path = img_path / (str(row['id']) + '.png')

        label = classes[str(row['label'])]
        image = decode_image(img_path)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [10]:
from torchvision.datasets import CIFAR10

mean, std = [0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]

train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=mean, std=std),
])

eval_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=mean, std=std),
])

data_train = ICIFAR10(labels_train, train_dir, transform=train_transform)
data_validation = ICIFAR10(labels_validation, validation_dir, transform=eval_transform)
data_test = CIFAR10(root= root_dir / 'cifar-10-python', train=False, download=False, transform=eval_transform)

## Fixing seed

In [11]:
import random
import numpy as np

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Dataloader

In [12]:
set_seed()

g_train = torch.Generator()
g_train.manual_seed(SEED)

train_dataloader = DataLoader(data_train, batch_size=64, shuffle=True, generator=g_train)
validation_dataloader = DataLoader(data_validation, batch_size=64)
test_dataloader = DataLoader(data_test, batch_size=64)

## SMOTE

In [13]:
from imblearn.over_sampling import SMOTE

In [14]:
Xs, ys = [], []

for X, y in train_dataloader:
    Xs.append(X.numpy()); ys.append(y.numpy())

X_train = np.concatenate(Xs).reshape(len(data_train), -1).astype(np.float32)
y_train = np.concatenate(ys)

sm = SMOTE(random_state=SEED, k_neighbors=5)
X_res, y_res = sm.fit_resample(X_train, y_train)

X_res = X_res.reshape(-1, 3, 32, 32).astype(np.float32)

class ArrayDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X, self.y, self.transform = X, y, transform
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        img = torch.from_numpy(self.X[i])
        if self.transform: img = self.transform(img)
        return img, int(self.y[i])

data_train_smote = ArrayDataset(X_res, y_res)

## Loading/Manipulating ResNet

In [15]:
set_seed()

# Load
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False)

# Manipulate
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(512, 10)
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): Identity()
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), p

Using cache found in C:\Users\Lucas/.cache\torch\hub\pytorch_vision_v0.10.0
D:\miniconda3\envs\gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\miniconda3\envs\gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


## Hyperparams setup

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using {device} device")

Using cuda device


In [17]:
import torch.optim as optim

epochs = 50
patience = 0
best_loss = float('inf')
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## Pondered loss

In [18]:
w_val = torch.tensor(1.0 / np.array(validation_sizes, dtype=np.float32)).to(device)
loss_pond_fn  = nn.CrossEntropyLoss(weight=w_val, reduction='sum')

## Training loop

In [17]:
set_seed()

for i in range(epochs):
    print(f'Época {i+1}/{epochs}')
    model.train()
    batch = 0
    for X,y in train_dataloader:
        if batch % 50 == 0:
            print(f'  batch {batch+1}/{len(train_dataloader)}')
        batch += 1

        X = X.to(device)
        y = y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    model.eval()
    total_loss = 0
    total_weights = 0
    with torch.no_grad():
        for X,y in validation_dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model(X)
            loss = loss_pond_fn(pred, y)

            total_loss += loss.item()
            total_weights += w_val[y].sum().item()

        total_loss = total_loss / total_weights
        if(total_loss < best_loss):
            best_loss = total_loss
            torch.save(model.state_dict(), root_dir / 'icifar10.pth')
            patience = 0
        else:
            patience += 1
            if patience == 15:
                print('Early stopping...')
                break
    print(f'Fim da época {i+1}: val_loss = {total_loss}')

Época 1/50
  batch 1/454
  batch 51/454
  batch 101/454
  batch 151/454
  batch 201/454
  batch 251/454
  batch 301/454
  batch 351/454
  batch 401/454
  batch 451/454
Fim da época 1: val_loss = 1.8101573153951864
Época 2/50
  batch 1/454
  batch 51/454
  batch 101/454
  batch 151/454
  batch 201/454
  batch 251/454
  batch 301/454
  batch 351/454
  batch 401/454
  batch 451/454
Fim da época 2: val_loss = 1.1452549236072045
Época 3/50
  batch 1/454
  batch 51/454
  batch 101/454
  batch 151/454
  batch 201/454
  batch 251/454
  batch 301/454
  batch 351/454
  batch 401/454
  batch 451/454
Fim da época 3: val_loss = 1.0683366845610116
Época 4/50
  batch 1/454
  batch 51/454
  batch 101/454
  batch 151/454
  batch 201/454
  batch 251/454
  batch 301/454
  batch 351/454
  batch 401/454
  batch 451/454
Fim da época 4: val_loss = 0.901944563903121
Época 5/50
  batch 1/454
  batch 51/454
  batch 101/454
  batch 151/454
  batch 201/454
  batch 251/454
  batch 301/454
  batch 351/454
  batch 4

## Loading best model

In [19]:
model.load_state_dict(torch.load(root_dir / 'icifar10.pth', weights_only=True))
model.eval()
print("",end="")

## Accuracy

In [20]:
y_true = []
y_pred = []
with torch.no_grad():
    acc = 0
    for X,y in test_dataloader:
        X = X.to(device)
        y = y.to(device)

        pred = model(X)
        pred_argmax = pred.argmax(1)
        acc += (pred_argmax == y).sum().item()

        for i in range(len(y)):
            y_true.append(y[i].item())
            y_pred.append(pred_argmax[i].item())

    acc = acc / len(data_test)

print(f'Acurácia: {(acc*100):.2f}%')

Acurácia: 70.95%


## Classification report

In [21]:
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=list(classes.keys()))
print(report)

              precision    recall  f1-score   support

    airplane       0.71      0.87      0.78      1000
  automobile       0.86      0.94      0.90      1000
        bird       0.65      0.56      0.60      1000
         cat       0.37      0.86      0.52      1000
        deer       0.75      0.65      0.70      1000
         dog       0.94      0.06      0.11      1000
        frog       0.87      0.75      0.80      1000
       horse       0.87      0.72      0.79      1000
        ship       0.91      0.83      0.87      1000
       truck       0.88      0.86      0.87      1000

    accuracy                           0.71     10000
   macro avg       0.78      0.71      0.69     10000
weighted avg       0.78      0.71      0.69     10000



## Using WeightedRandomSampler

In [21]:
targets = [classes[str(l)] for l in data_train.img_labels['label']]

In [22]:
print(f'[Targets] {targets[:10]}...')
print(f'[Class counts] {train_sizes}')

[Targets] [9, 9, 1, 7, 3, 7, 7, 2, 9, 9]...
[Class counts] [4500, 3080, 2385, 3497, 2718, 205, 2273, 3877, 2033, 4441]


In [23]:
class_weights = 1.0 / np.array(train_sizes, dtype=np.float64)
print(f'[Class weights] {class_weights}')

[Class weights] [0.00022222 0.00032468 0.00041929 0.00028596 0.00036792 0.00487805
 0.00043995 0.00025793 0.00049188 0.00022517]


In [24]:
sample_weights = class_weights[targets]
print(f'[Sample weights] {sample_weights}')

[Sample weights] [0.00022517 0.00022517 0.00032468 ... 0.00032468 0.00022222 0.00032468]


In [25]:
from torch.utils.data import WeightedRandomSampler

set_seed()

g_sampler = torch.Generator()
g_sampler.manual_seed(SEED)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
    generator=g_sampler
)

train_sampler_dataloader = DataLoader(
    dataset=data_train,
    batch_size=64,
    sampler=sampler
)

In [26]:
set_seed()

model_sampler = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False)
model_sampler.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_sampler.maxpool = nn.Identity()
model_sampler.fc = nn.Linear(512, 10)
model_sampler = model_sampler.to(device)

Using cache found in C:\Users\Lucas/.cache\torch\hub\pytorch_vision_v0.10.0
D:\miniconda3\envs\gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\miniconda3\envs\gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


## Using SMOTE

In [22]:
set_seed()
g_smote = torch.Generator(); g_smote.manual_seed(SEED)

train_smote_dataloader = DataLoader(data_train_smote, batch_size=64, shuffle=True, generator=g_smote)
model_smote = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=False)
model_smote.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_smote.maxpool = nn.Identity()
model_smote.fc = nn.Linear(512, 10)
model_smote = model_smote.to(device)

Using cache found in C:\Users\Lucas/.cache\torch\hub\pytorch_vision_v0.10.0
D:\miniconda3\envs\gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\miniconda3\envs\gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [23]:
epochs = 50
patience = 0
best_loss = float('inf')
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_smote.parameters(), lr=1e-3)

In [24]:
set_seed()

for i in range(epochs):
    print(f'Época {i+1}/{epochs}')
    model_smote.train()
    batch = 0
    for X,y in train_smote_dataloader:
        if batch % 50 == 0:
            print(f'  batch {batch+1}/{len(train_smote_dataloader)}')
        batch += 1

        X = X.to(device)
        y = y.to(device)

        # Compute prediction error
        pred = model_smote(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    model_smote.eval()
    total_loss = 0
    total_weights = 0
    with torch.no_grad():
        for X,y in validation_dataloader:
            X = X.to(device)
            y = y.to(device)

            pred = model_smote(X)
            loss = loss_pond_fn(pred, y)

            total_loss += loss.item()
            total_weights += w_val[y].sum().item()

        total_loss = total_loss / total_weights
        if(total_loss < best_loss):
            best_loss = total_loss
            torch.save(model_smote.state_dict(), root_dir / 'icifar10_smote.pth')
            patience = 0
        else:
            patience += 1
            if patience == 15:
                print('Early stopping...')
                break
    print(f'Fim da época {i+1}: val_loss = {total_loss}')

Época 1/50
  batch 1/704
  batch 51/704
  batch 101/704
  batch 151/704
  batch 201/704
  batch 251/704
  batch 301/704
  batch 351/704
  batch 401/704
  batch 451/704
  batch 501/704
  batch 551/704
  batch 601/704
  batch 651/704
  batch 701/704
Fim da época 1: val_loss = 1.4437721723922634
Época 2/50
  batch 1/704
  batch 51/704
  batch 101/704
  batch 151/704
  batch 201/704
  batch 251/704
  batch 301/704
  batch 351/704
  batch 401/704
  batch 451/704
  batch 501/704
  batch 551/704
  batch 601/704
  batch 651/704
  batch 701/704
Fim da época 2: val_loss = 1.2489803250917035
Época 3/50
  batch 1/704
  batch 51/704
  batch 101/704
  batch 151/704
  batch 201/704
  batch 251/704
  batch 301/704
  batch 351/704
  batch 401/704
  batch 451/704
  batch 501/704
  batch 551/704
  batch 601/704
  batch 651/704
  batch 701/704
Fim da época 3: val_loss = 1.245766528525328
Época 4/50
  batch 1/704
  batch 51/704
  batch 101/704
  batch 151/704
  batch 201/704
  batch 251/704
  batch 301/704

In [25]:
model_smote.load_state_dict(torch.load(root_dir / 'icifar10_smote.pth', weights_only=True))
model_smote.eval()
print("",end="")

In [26]:
y_true = []
y_pred = []
with torch.no_grad():
    acc = 0
    for X,y in test_dataloader:
        X = X.to(device)
        y = y.to(device)

        pred = model_smote(X)
        pred_argmax = pred.argmax(1)
        acc += (pred_argmax == y).sum().item()

        for i in range(len(y)):
            y_true.append(y[i].item())
            y_pred.append(pred_argmax[i].item())

    acc = acc / len(data_test)

print(f'Acurácia: {(acc*100):.2f}%')

Acurácia: 71.51%


In [27]:
report = classification_report(y_true, y_pred, target_names=list(classes.keys()))
print(report)

              precision    recall  f1-score   support

    airplane       0.59      0.89      0.71      1000
  automobile       0.85      0.91      0.88      1000
        bird       0.59      0.71      0.64      1000
         cat       0.51      0.60      0.55      1000
        deer       0.76      0.64      0.70      1000
         dog       0.76      0.17      0.27      1000
        frog       0.84      0.76      0.79      1000
       horse       0.73      0.74      0.74      1000
        ship       0.88      0.85      0.87      1000
       truck       0.81      0.89      0.85      1000

    accuracy                           0.72     10000
   macro avg       0.73      0.72      0.70     10000
weighted avg       0.73      0.72      0.70     10000

